In [1]:
import os, time, sys
from pathlib import Path

import numpy as np
import nibabel as nib
import torch


RUN_DIRS = [
    Path("/home/ayeluru/mnt/fourier/projects/vascular-superenhancement-4d-flow/all_patients/runs/2026-02-18/02-08-55"),
    Path("/home/ayeluru/mnt/fourier/projects/vascular-superenhancement-4d-flow/all_patients/runs/2026-02-14/01-02-34"),
]

NIFTI_EXTS = (".nii", ".nii.gz")
CKPT_EXTS  = (".pt", ".pth", ".ckpt")  # adjust if you use something else


def iter_files(root: Path):
    for dp, _, fs in os.walk(root):
        for f in fs:
            yield Path(dp) / f


def force_read_nifti(p: Path):
    img = nib.load(str(p))
    arr = np.asanyarray(img.dataobj)  # forces full payload read
    if arr.size:
        _ = float(arr.flat[0])
        _ = float(arr.flat[arr.size // 2])
        _ = float(arr.flat[-1])


def check_gzip_integrity(p: Path):
    # For .nii.gz and any other gz files (wandb files, logs, etc)
    import gzip
    with gzip.open(p, "rb") as f:
        # stream through without holding in memory
        for _ in iter(lambda: f.read(1024 * 1024), b""):
            pass


def check_checkpoint(p: Path):
    # CPU-only load; catches truncation/pickle errors.
    obj = torch.load(p, map_location="cpu")
    # lightweight touch: common keys if it's a dict
    if isinstance(obj, dict):
        _ = list(obj.keys())[:5]


def run_checks(root: Path):
    print(f"\n========== CHECKING: {root} ==========")
    if not root.exists():
        print("MISSING DIRECTORY")
        return

    t0 = time.time()

    n_nifti = n_ckpt = n_gz = 0
    bad_nifti = bad_ckpt = bad_gz = 0

    for i, p in enumerate(iter_files(root), start=1):
        suffix = p.suffix.lower()
        name = p.name.lower()

        # NIfTI full read
        if name.endswith(NIFTI_EXTS):
            n_nifti += 1
            try:
                force_read_nifti(p)
            except Exception as e:
                bad_nifti += 1
                print(f"[BAD NIFTI] {p}\n    {type(e).__name__}: {e}")
                sys.stdout.flush()

        # gzip integrity check for anything *.gz (including nii.gz)
        if name.endswith(".gz"):
            n_gz += 1
            try:
                check_gzip_integrity(p)
            except Exception as e:
                bad_gz += 1
                print(f"[BAD GZ] {p}\n    {type(e).__name__}: {e}")
                sys.stdout.flush()

        # checkpoint load sanity
        if name.endswith(CKPT_EXTS):
            n_ckpt += 1
            try:
                check_checkpoint(p)
            except Exception as e:
                bad_ckpt += 1
                print(f"[BAD CKPT] {p}\n    {type(e).__name__}: {e}")
                sys.stdout.flush()

        if i % 200 == 0:
            print(
                f"[PROGRESS] files={i} | nifti {n_nifti} bad {bad_nifti} | "
                f"gz {n_gz} bad {bad_gz} | ckpt {n_ckpt} bad {bad_ckpt}"
            )
            sys.stdout.flush()

    dt = time.time() - t0
    print("\n----- SUMMARY -----")
    print(f"nifti checked: {n_nifti} | bad: {bad_nifti}")
    print(f"gzip checked : {n_gz} | bad: {bad_gz}")
    print(f"ckpt checked : {n_ckpt} | bad: {bad_ckpt}")
    print(f"elapsed: {dt:.1f}s")


for rd in RUN_DIRS:
    run_checks(rd)


========== CHECKING: /home/ayeluru/mnt/fourier/projects/vascular-superenhancement-4d-flow/all_patients/runs/2026-02-18/02-08-55 ==========


/tmp/ipykernel_1258251/129713088.py:44: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  obj = torch.load(p, map_location="cpu")


[PROGRESS] files=200 | nifti 177 bad 0 | gz 177 bad 0 | ckpt 15 bad 0
[PROGRESS] files=400 | nifti 376 bad 0 | gz 376 bad 0 | ckpt 15 bad 0
[PROGRESS] files=600 | nifti 576 bad 0 | gz 576 bad 0 | ckpt 15 bad 0
[PROGRESS] files=800 | nifti 776 bad 0 | gz 776 bad 0 | ckpt 15 bad 0
[PROGRESS] files=1000 | nifti 976 bad 0 | gz 976 bad 0 | ckpt 15 bad 0
[PROGRESS] files=1200 | nifti 1176 bad 0 | gz 1176 bad 0 | ckpt 15 bad 0
[PROGRESS] files=1400 | nifti 1376 bad 0 | gz 1376 bad 0 | ckpt 15 bad 0
[PROGRESS] files=1600 | nifti 1576 bad 0 | gz 1576 bad 0 | ckpt 15 bad 0

----- SUMMARY -----
nifti checked: 1601 | bad: 0
gzip checked : 1601 | bad: 0
ckpt checked : 15 | bad: 0
elapsed: 230.3s

========== CHECKING: /home/ayeluru/mnt/fourier/projects/vascular-superenhancement-4d-flow/all_patients/runs/2026-02-14/01-02-34 ==========
[PROGRESS] files=200 | nifti 164 bad 0 | gz 164 bad 0 | ckpt 29 bad 0
[PROGRESS] files=400 | nifti 364 bad 0 | gz 364 bad 0 | ckpt 29 bad 0
[PROGRESS] files=600 | nifti